In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from bs4 import BeautifulSoup
import re
import pdfplumber
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
import tabula
import requests

from pathlib import Path
import chardet

import datetime
from selenium import webdriver
from time import sleep
import os
from selenium.webdriver.chrome.service import Service as ChromeService



from zipfile import ZipFile
import gzip




In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'NZ CIFSC' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running NZ CIFSC Web Scraping Tool v.1.1


In [3]:
#------------------------------------------------ Begin_chromedriver ----------------------------------------
#Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
		 "download.prompt_for_download": False,
		 "download.default_directory" : tempfolder,
         'profile.default_content_setting_values.automatic_downloads': 1 # Desable a Multiplefile download alert
         }
chromeOptions.add_experimental_option("prefs",prefs)
driver = webdriver.Chrome(options=chromeOptions)
driver.maximize_window()


In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict={

         regulatorName + ' 1': 'https://www.fsc.gov.ck/public/content.aspx?cn=Banks',
         regulatorName + ' 2': 'https://www.fsc.gov.ck/public/content.aspx?cn=insurers',
         regulatorName + ' 3': 'https://www.fsc.gov.ck/public/content.aspx?cn=moneychanging',
        regulatorName + ' 4': 'https://www.fsc.gov.ck/public/content.aspx?cn=trusteecompanies',

        }



Typology={

       regulatorName + ' 1': 'List of Banks',
       regulatorName + ' 2': 'List of Insurers',
       regulatorName + ' 3': 'List of Money Changing',
       regulatorName + ' 4': 'List of Trustee Companies',

        }





In [6]:
#------------------------------------------------ Begin_Fouction ----------------------------------------
def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict

In [7]:

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/143.0.0.0 Safari/537.36"
        }
    resp = requests.get(regdict[reg], headers=headers, timeout=30,verify=False)
    resp.raise_for_status()
    print(resp.url)
    print(resp.status_code)
    soup = BeautifulSoup(resp.text, "html.parser")
    toc = soup.find(class_="content-toc-container")
    if reg == regulatorName + ' 1' or reg == regulatorName + ' 4':
        headers = toc.find_all("h4", limit=2)
        for h4 in headers:
            # walk forward until the next h4
            for sib in h4.find_next_siblings():
                if sib.name == "h4":
                    break
                if sib.name != "p":
                    continue
                a = sib.find("a", href=True)
                if a and a["href"].startswith("http"):
                    name_ = a.get_text(strip=True)
                    website_ = a["href"]
                    cate_name = h4.get_text(strip=True)
                    #print(name_, website_, cate_name)
                    sqldict['Name'].append(name_.replace('*','').strip())
                    sqldict['Website'].append(website_)
                    sqldict['Typology'].append(cate_name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)

    elif reg == regulatorName + ' 2' or reg == regulatorName + ' 3':
        results = []
        headers = toc.find_all("strong")
        for strong in headers:
            # walk forward until the next h4
            if reg == regulatorName + ' 2' :
                cat_name = strong.get_text(strip=True).rstrip(":")
                companies = []

                # walk forward until next strong or h4
                for sib in strong.parent.find_next_siblings():
                    if sib.find("strong") or sib.name == "h4":
                        break

                    text = sib.get_text(" ", strip=True)
                    # pick only bullet-style company lines
                    if "*" in text:
                        # remove bullet and extra spaces
                        name = re.sub(r"^[*\s\u00a0]+", "", text.replace("*", ""))
                        name = re.sub(r"\s+", " ", name).strip()
                        if name:
                            companies.append(name)
            elif reg == regulatorName + ' 3' :
                header_p = strong.find_parent("p")
                if not header_p:
                    continue

                cat_name = strong.get_text(strip=True).rstrip(":")
                companies = []

                for sib in header_p.find_next_siblings():
                    if sib.find("strong") or sib.name == "h4":
                        break

                    text = sib.get_text(" ", strip=True)
                    if text.startswith("*"):
                        name = re.sub(r"^\*\s*", "", text).strip()
                        if name:
                            companies.append(name)

            results.append({"cat_name": cat_name, "companies": companies})

        for r in results:
            for c in r["companies"]:
                # print(r["cat_name"] + "  -", c)
                sqldict['Name'].append(c)
                cate_name = r["cat_name"]
                sqldict['Typology'].append(cate_name.removesuffix('are'))
                sqldict['ListProcessDate'].append(processdate)
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['RegulationType'].append('Regulated')
                sqldict['ListName'].append(Typology[reg])
                sqldict = bourange_same_length_array(sqldict)                

[INFO] : Working 1/4 _(NZ CIFSC 1)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.fsc.gov.ck'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://www.fsc.gov.ck/public/content.aspx?cn=Banks
200
[INFO] : Working 2/4 _(NZ CIFSC 2)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.fsc.gov.ck'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://www.fsc.gov.ck/public/content.aspx?cn=insurers
200
[INFO] : Working 3/4 _(NZ CIFSC 3)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.fsc.gov.ck'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://www.fsc.gov.ck/public/content.aspx?cn=moneychanging
200
[INFO] : Working 4/4 _(NZ CIFSC 4)_ 


c:\Users\wuj1\AppData\Local\Programs\Python\Python312\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.fsc.gov.ck'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


https://www.fsc.gov.ck/public/content.aspx?cn=trusteecompanies
200


In [8]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
# df.to_excel(filename, index=False)
# driver.quit()
# sleep(3)

In [9]:

import json
import requests
API_TOKEN = ""
URL = "https://Api.bvdinfo.com/v1/orbis/companies/match"



headers = {
    "ApiToken": API_TOKEN,
    "Content-Type": "application/json"
}
# df is your dataframe with a "Name" column
for i, row in df.iterrows():
    ITERATE_NAME = str(row["Name"]).strip() if row["Name"] is not None else ""
    # fax_ = str(row["Fax"]).strip() if row["Fax"] is not None else ""
    # tel_ = str(row["Phone"]).strip() if row["Phone"] is not None else ""
    # address_ = str(row["Address_1"]).strip() if row["Address_1"] is not None else ""
    # email_ = str(row["Email"]).strip() if row["Email"] is not None else ""
    website_ = str(row["Website"]).strip().removeprefix('https://') if row["Website"] is not None else ""
    payload = {
        "MATCH": {
            "Criteria": {
                "Name": ITERATE_NAME,
                "Country": "NZ",
                # "EMailOrWebsite": email_ or website_,
                # "Address": address_,
                # "PhoneOrFax": fax_ or tel_
            },
            "Options": {
                "ScoreLimit": 0.85
            }
        },
        "SELECT": [
            "Match.Hint",
            "Match.Score",
            "Match.Name",
            "Match.Name_Local",
            "Match.Address",
            "Match.Postcode",
            "Match.City",
            "Match.Country",
            "Match.Status",
            "Match.National_Id",
            "Match.NationalIdLabel",
            "Match.LegalForm",
            "Match.BvDId",
            "Match.PhoneOrFax",
            "Match.EmailOrWebsite"
        ]
    }

    response = requests.post(URL, headers=headers, json=payload, timeout=60)
    #print("Status:", response.status_code)

    try:
        data = response.json()
        bvd_id = ""
        print(json.dumps(data, indent=2))
        if data and data[0].get("Hint", "").lower() != "unlikely":
            if len(data) > 1:
                print(f"Multiple matches found for row {i}: {len(data)} matches")
                top_score = data[0].get("Score", 0)
                tied = [d for d in data if d.get("Score", 0) == top_score]

                if len(tied) > 1:
                    words = [w for w in ITERATE_NAME.split() if w]
                    def hint_name_count(d):
                        name = d.get("Name", "").lower()
                        return sum(1 for w in words if w.lower() in name)

                    best = max(tied, key=hint_name_count)  # first wins on tie
                    print(best)
                    data[0] = best


            bvd_id = data[0].get("BvDId", "")
    except Exception:
        #print(response.text)
        bvd_id = ""
    #print(bvd_id)
    df.at[i, "bvdid"] = bvd_id if bvd_id else ""



[
  {
    "Hint": "Potential",
    "Score": 0.95,
    "Name": "ANZ BANK NEW ZEALAND LIMITED",
    "Name_Local": null,
    "Address": "GROUND FLOOR, ANZ CENTRE, 23-29 ALBERT STREET",
    "Postcode": "1010",
    "City": "AUCKLAND",
    "Country": "NZ",
    "Status": "Active",
    "National_Id": "35976",
    "NationalIdLabel": "Company number",
    "LegalForm": "Corporation",
    "BvDId": "NZ9429040797410",
    "PhoneOrFax": "+64 4 470 31 42",
    "EmailOrWebsite": "www.anz.co.nz"
  },
  {
    "Hint": "Potential",
    "Score": 0.95,
    "Name": "ANZ BANKING GROUP (NEW ZEALAND) LIMITED",
    "Name_Local": null,
    "Address": "L 15 ANZ TOWER 215-229 LAMBTON QUAY WELLINGTON",
    "Postcode": "6149",
    "City": "WELLINGTON",
    "Country": "NZ",
    "Status": "Active",
    "National_Id": null,
    "NationalIdLabel": null,
    "LegalForm": null,
    "BvDId": "NZ*110371548909",
    "PhoneOrFax": null,
    "EmailOrWebsite": "www.anz.co.nz/personal"
  },
  {
    "Hint": "Unlikely",
    "Score":

In [ ]:
df.to_excel(filename, index=False)
driver.quit()
sleep(3)